# Notebook 6 — Envío de predicciones al PLC Beckhoff (ADS)
## Del clasificador a la acción del brazo

Último paso de la **Parte 2 (Visión)**. El notebook **solo envía datos** al IPC Beckhoff
por ADS; el **HMI con las lámparas y la lógica del brazo se programan en TwinCAT**.

Avanzamos de lo simple a lo completo:

| Etapa | Qué se hace |
|------|-------------|
| 1–2 | Setup y conexión ADS |
| **3. Enviar booleanos** | Prender/apagar `bRojo/bAzul/bAmarillo` y verlos en tus lámparas de TwinCAT |
| 4. Predicción → envío | Clasificar **una** imagen y enviar el resultado |
| 5. (Extra) Cámara | Clasificar en vivo y enviar al pulsar una tecla |

> **Antes (en TwinCAT):** declara 3 variables `BOOL` (`bRojo`, `bAzul`, `bAmarillo`) en una
> GVL, arma un HMI con una lámpara por color y pon TwinCAT en **Run**. Anota el **AMS Net ID**.
>
> ⚠️ **El código ADS es de referencia** (no probado con tu hardware): puede requerir ajustes
> o fallar; completarlo/validarlo es parte del trabajo. Docs: https://pyads.readthedocs.io
> · https://github.com/stlehmann/pyads . Sin PLC, el notebook corre en **modo simulación**.

## ✅ Qué debes completar

Casi todo el código ya está listo. **Solo necesitas tocar 3 cosas** (las marcadas con
`# COMPLETAR` en la celda de configuración):

1. **En TwinCAT** (fuera del notebook): declarar `bRojo`, `bAzul`, `bAmarillo` en una GVL,
   armar el HMI con las lámparas y poner TwinCAT en **Run**.
2. **`AMS_NET_ID`** → el de tu IPC Beckhoff.
3. **`VAR_POR_COLOR`** → solo si tu GVL no se llama `VARIABLES` (ajusta el prefijo).

El resto **no lo modifiques** para la prueba: las funciones de envío, predicción y cámara
ya funcionan. (Opcional: `UMBRAL` y el índice de cámara `cv2.VideoCapture(0)`.)

---
## 1. Setup y configuración

In [ ]:
import os
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '-1')
import glob, time
import json as json_lib
from collections import deque
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

try:
    import pyads
    PYADS_OK = True
except ImportError:
    PYADS_OK = False

# ══════════════════════════════════════════════════════════════════
#  CONFIGURACIÓN  —  esto es LO ÚNICO que debes completar
# ══════════════════════════════════════════════════════════════════

# COMPLETAR (1): AMS Net ID de TU IPC Beckhoff (lo ves en TwinCAT)
AMS_NET_ID = '5.80.201.232.1.1'
ADS_PORT   = 851                  # 851 = TwinCAT 3 Runtime 1 (normalmente no se cambia)

# COMPLETAR (2): nombres EXACTOS de tus variables BOOL en TwinCAT.
#   Cambia el prefijo 'VARIABLES.' solo si tu GVL se llama distinto.
VAR_POR_COLOR = {
    'rojo':     'VARIABLES.bRojo',
    'azul':     'VARIABLES.bAzul',
    'amarillo': 'VARIABLES.bAmarillo',
}

# (Opcional) confianza minima para enviar; subela/bajala segun tu modelo:
UMBRAL = 0.60
# ══════════════════════════════════════════════════════════════════
print('pyads:', 'OK' if PYADS_OK else 'no instalado (modo simulacion)')

---
## 2. Conexión con el PLC

Necesitas **TwinCAT en Run**, el **AMS Net ID** correcto y una **ruta ADS** entre tu PC y el
target. Si falta la ruta verás `Missing ADS routes (7)`: créala en
**TwinCAT (bandeja) → Router → Edit Routes → Add Route**. Si algo falla, el notebook sigue
en **modo simulación** sin detenerse.

In [ ]:
# Código de referencia — adáptalo a tu setup. Docs: https://pyads.readthedocs.io
plc = {'conn': None, 'connected': False}

def conectar():
    if not PYADS_OK:
        print('pyads no instalado  ->  MODO SIMULACION'); return
    try:
        conn = pyads.Connection(AMS_NET_ID, ADS_PORT)
        conn.open()
        conn.read_state()              # comprueba que la RUTA ADS funcione
        plc['conn'] = conn; plc['connected'] = True
        print(f'Conectado a {AMS_NET_ID}:{ADS_PORT}')
    except Exception as e:
        plc['connected'] = False
        print(f'No se pudo conectar ({e})  ->  MODO SIMULACION')
        print('Falta la ruta ADS o el AMS Net ID. Crea la ruta en TwinCAT > Router > Edit Routes.')

conectar()

---
## 3. Prueba: enviar booleanos

La prueba clave: **¿puedo enviar datos al PLC?** Ejecuta las líneas de abajo y observa cómo
se prende/apaga la **lámpara correspondiente en tu HMI de TwinCAT**. `enviar_color` activa el
BOOL de un color y apaga los otros dos; `apagar_todo()` los deja en `False`.

In [ ]:
def escribir_bool(variable, valor):
    """Escribe un BOOL en el PLC (si hay conexion)."""
    if not plc['connected']:
        return
    try:
        plc['conn'].write_by_name(variable, bool(valor), pyads.PLCTYPE_BOOL)
    except Exception as e:
        print('Error ADS:', e, ' ->  paso a modo simulacion')
        plc['connected'] = False

def enviar_color(color):
    """Activa el BOOL del color (rojo/azul/amarillo) y apaga los demas. None = apagar todo."""
    for c, variable in VAR_POR_COLOR.items():
        escribir_bool(variable, c == color)
    canal = 'PLC' if plc['connected'] else 'SIM'
    print(f'[{canal}] enviado: {color if color else "apagar todo"}')

def apagar_todo():
    enviar_color(None)

# PRUEBA: ejecuta una linea a la vez y mira la lampara en TwinCAT
enviar_color('rojo')
# enviar_color('azul')
# enviar_color('amarillo')
# apagar_todo()

---
## 4. Predicción → envío al PLC

Carga el modelo entrenado en la Parte 2, clasifica **una** imagen y envía el color detectado.
Solo se envía si la confianza supera `UMBRAL` y el color tiene una salida (la clase `fondo`
nunca envía).

In [ ]:
# --- Cargar el modelo mas reciente de modelos/ ---
rutas = glob.glob('modelos/*.h5') or glob.glob('../modelos/*.h5')
MODEL_PATH = max(rutas, key=os.path.getmtime)   # el entrenado mas recientemente
# (si quieres otro modelo, escribe su ruta a mano: MODEL_PATH = 'modelos/tu_modelo.h5')
modelo = load_model(MODEL_PATH, compile=False)
info = json_lib.load(open(MODEL_PATH.rsplit('.', 1)[0] + '.json'))
CLASES = info['class_names']                 # p.ej. ['amarillo','azul','fondo','rojo']
IMG_SIZE = info['img_size']
PREP = info['preprocessing']                 # 'mobilenet' o 'rescale'
print('Modelo:', os.path.basename(MODEL_PATH), '| clases:', CLASES, '| tamano:', IMG_SIZE)

def preparar(imagen_rgb):
    """Normaliza una imagen RGB segun el tipo de modelo."""
    x = imagen_rgb.astype('float32')
    return preprocess_input(x) if PREP == 'mobilenet' else x / 255.0

def predecir(ruta_imagen, mostrar=True):
    """Devuelve (clase, confianza) para una imagen y opcionalmente la muestra."""
    img = load_img(ruta_imagen, target_size=(IMG_SIZE, IMG_SIZE))
    pred = modelo.predict(np.expand_dims(preparar(img_to_array(img)), 0), verbose=0)[0]
    idx = int(np.argmax(pred))
    clase, conf = CLASES[idx], float(pred[idx])
    if mostrar:
        plt.imshow(img); plt.axis('off'); plt.title(f'{clase}  ({conf:.0%})'); plt.show()
    return clase, conf

def enviar_prediccion(ruta_imagen):
    """Predice una imagen y envia el color al PLC si corresponde."""
    clase, conf = predecir(ruta_imagen)
    if clase not in VAR_POR_COLOR:
        print(f'No se envia: "{clase}" no tiene salida (p.ej. fondo).')
    elif conf < UMBRAL:
        print(f'No se envia: confianza {conf:.0%} < umbral {UMBRAL:.0%} '
              f'(acerca la tapita o baja UMBRAL).')
    else:
        enviar_color(clase)

In [ ]:
# Prueba con una imagen del dataset (cambia la ruta por una foto tuya si quieres)
ejemplos = glob.glob('data/tapitas/rojo/*') or glob.glob('../data/tapitas/rojo/*')
enviar_prediccion(ejemplos[0])

---
## 5. (Extra) Cámara en vivo → PLC

La cámara clasifica en tiempo real y muestra la predicción. **Por defecto solo envía al
pulsar `ESPACIO`** (envío consciente). En pantalla ves la predicción y el último envío.

**Controles:** `ESPACIO` = enviar · `Q` = salir.

> En **WSL2** no hay cámara USB: ejecuta este notebook desde **Windows**.

In [ ]:
def es_wsl():
    try: return 'microsoft' in open('/proc/version').read().lower()
    except Exception: return False

if es_wsl():
    print('WSL2 no accede a la camara. Ejecuta este notebook desde Windows.')
else:
    cam = cv2.VideoCapture(0)          # cambia a 1 si no detecta la camara
    historial = deque(maxlen=5)        # promedia 5 frames (anti-parpadeo)
    ultimo_envio = '(nada)'
    print('Camara abierta.  ESPACIO = enviar  |  Q = salir')
    try:
        while True:
            ok, frame = cam.read()
            if not ok:
                break

            # Predecir el frame (RGB -> preparar -> modelo)
            rgb = cv2.cvtColor(cv2.resize(frame, (IMG_SIZE, IMG_SIZE)), cv2.COLOR_BGR2RGB)
            pred = modelo.predict(np.expand_dims(preparar(rgb), 0), verbose=0)[0]
            historial.append(pred)
            media = np.mean(historial, axis=0)
            clase = CLASES[int(np.argmax(media))]
            conf = float(np.max(media))

            # Mostrar prediccion y ultimo envio
            cv2.putText(frame, f'{clase} ({conf*100:.0f}%)', (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            cv2.putText(frame, f'Ultimo envio: {ultimo_envio}', (10, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 255), 2)
            cv2.imshow('Camara -> PLC (ESPACIO=enviar, Q=salir)', frame)

            tecla = cv2.waitKey(1) & 0xFF
            if tecla == ord('q'):
                break
            if tecla == ord(' '):          # enviar SOLO al pulsar ESPACIO
                if clase in VAR_POR_COLOR and conf >= UMBRAL:
                    enviar_color(clase)
                    ultimo_envio = clase
                else:
                    enviar_color(None)
                    ultimo_envio = 'ninguno (fondo o baja confianza)'
    finally:
        apagar_todo()
        cam.release()
        cv2.destroyAllWindows()
        print('Camara cerrada y BOOLs en False.')